# Public-data experiments

Run from the repository root with the dependencies in `requirements.txt`. Execute cells in order. The configuration in `configs/experiment.json` defines the experiment. Training cells run the full pipeline; pilot outputs only validate the execution path.


In [ ]:
from pathlib import Path
import sys, subprocess, json
assert Path('configs/experiment.json').exists(), 'Postavi radni direktorijum na koren paketa'
def run(*args):
    subprocess.run([sys.executable, *args], check=True)


## Prepare CICIDS2017 Improved


In [ ]:
run('scripts/download_data.py', '--dataset', 'cic2017', '--root', 'work')
run('-m','ids','prepare','--source-root','work/data_sources','--dataset','cic2017','--output','work/prepared2017')
run('-m','ids','pilot','--data','work/prepared2017','--output','work/pilot2017')


## Prepare CSE-CIC-IDS2018 Improved


In [ ]:
run('scripts/download_data.py', '--dataset', 'cic2018', '--root', 'work')
run('-m','ids','prepare','--source-root','work/data_sources','--dataset','cic2018','--output','work/prepared2018')
run('-m','ids','pilot','--data','work/prepared2018','--output','work/pilot2018')


## Development and model selection

This stage performs 30 autoencoder fits and five Isolation Forest fits, then locks the architecture and threshold decision.


In [ ]:
run('-m','ids','develop','--data','work/prepared2017','--experiment','work/experiment')
decision=json.loads(Path('work/experiment/decision.json').read_text())
{k:decision[k] for k in ['architecture','percentile_ae','percentile_if']}


## Final evaluation and external replication


In [ ]:
run('-m','ids','final','--data','work/prepared2017','--experiment','work/experiment')
run('-m','ids','external','--data','work/prepared2018','--experiment','work/experiment')


In [ ]:
for flags in [[],['--external'],['--external','--diagnostic']]:
    for command in ['summarize','figures']:
        run('-m','ids',command,'--experiment','work/experiment',*flags)
